# Amazon Bedrock Text Generation

This notebook demonstrates basic text generation using Amazon Bedrock's invoke_model API. This is the fundamental building block for most generative AI applications, showing how to send prompts to foundation models and receive text responses.

## Key Concepts:
- **Model Invocation**: Direct API calls to foundation models
- **Request/Response Pattern**: Synchronous communication with models
- **Inference Configuration**: Parameters that control model behavior
- **Message Format**: Structured input format for conversational models

## Model Selection and Setup

Amazon Nova Micro is selected for this example because:
- **Cost-effective**: Lower pricing for simple text generation tasks
- **Fast response**: Optimized for quick inference
- **Good quality**: Suitable for most text transformation tasks

The `invoke_model` API is the standard method for synchronous model calls.

In [1]:
import boto3
import json

# Amazon Nova Micro - cost-effective model for text generation
MODEL_ID = "amazon.nova-micro-v1:0"
bedrock = boto3.client(service_name='bedrock-runtime', region_name='us-east-1')

# Example: Text style transformation
# This demonstrates how to use AI for text rewriting tasks
response = bedrock.invoke_model(
    modelId=MODEL_ID,
    guardrailVersion="DRAFT",  # Optional: Apply content filtering
    body=json.dumps({
        "schemaVersion": "messages-v1",  # Standard message format
        "messages": [{"role": "user", "content": [{"text": "Rewrite this sentence for me in a formal tone: You are very good at your job."}]}],
        "inferenceConfig": {
            "maxTokens": 500,      # Limit response length
            "topK": 20,           # Diversity control
            "temperature": 0.7    # Creativity vs consistency balance
        }
    })
)

# Raw response contains metadata and the generated text
response['body'].read()

b'{"output":{"message":{"content":[{"text":"You exhibit exceptional proficiency in your professional responsibilities."}],"role":"assistant"}},"stopReason":"end_turn","usage":{"inputTokens":19,"outputTokens":11,"totalTokens":30,"cacheReadInputTokenCount":0,"cacheWriteInputTokenCount":0}}'

## Understanding the Response Structure

The response from Bedrock contains several important components:

- **output.message.content**: The actual generated text
- **stopReason**: Why the model stopped generating (end_turn, max_tokens, etc.)
- **usage**: Token consumption metrics for cost tracking
- **ResponseMetadata**: HTTP headers with latency and token count information

### Inference Configuration Parameters:
- **maxTokens**: Maximum response length (cost control)
- **temperature**: 0.0 = deterministic, 1.0 = very creative
- **topK**: Number of top tokens to consider (diversity)
- **topP**: Cumulative probability threshold (alternative to topK)

In [6]:
# The complete response object includes metadata useful for monitoring
# Key metrics in headers:
# - invocation-latency: Response time in milliseconds
# - input-token-count: Tokens in the request (for billing)
# - output-token-count: Tokens in the response (for billing)
response

{'ResponseMetadata': {'RequestId': 'f8f6486e-b3f5-4098-b9c7-e474d3b1fb0d',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Fri, 19 Dec 2025 17:05:14 GMT',
   'content-type': 'application/json',
   'content-length': '286',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'f8f6486e-b3f5-4098-b9c7-e474d3b1fb0d',
   'x-amzn-bedrock-invocation-latency': '211',
   'x-amzn-bedrock-cache-write-input-token-count': '0',
   'x-amzn-bedrock-cache-read-input-token-count': '0',
   'x-amzn-bedrock-output-token-count': '11',
   'x-amzn-bedrock-input-token-count': '19'},
  'RetryAttempts': 0},
 'contentType': 'application/json',
 'body': <botocore.response.StreamingBody at 0x14985d8a0>}

## Best Practices for Text Generation

### Cost Optimization:
- Use appropriate models for the task complexity
- Set reasonable maxTokens limits
- Monitor token usage through response headers

### Quality Control:
- Adjust temperature based on use case (lower for factual, higher for creative)
- Use guardrails for content filtering
- Test different inference parameters

### Production Considerations:
- Implement error handling for API failures
- Add retry logic with exponential backoff
- Log requests and responses for debugging
- Consider async patterns for high-volume applications